Imports and global settings

In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
import lightgbm as lgb

Load train and test data

In [2]:
train = pd.read_csv("data/training_set_VU_DM.csv", na_values=["NULL"])
test_df = pd.read_csv("data/test_set_VU_DM.csv", na_values=["NULL"])

print("Train shape:", train.shape)
print("Test shape:", test_df.shape)

display(train.head())

Train shape: (4958347, 54)
Test shape: (4959183, 50)


,srch_id,date_time,site_id,visitor_location_country_id,visitor_hist_starrating,visitor_hist_adr_usd,prop_country_id,prop_id,prop_starrating,prop_review_score,...,comp6_rate_percent_diff,comp7_rate,comp7_inv,comp7_rate_percent_diff,comp8_rate,comp8_inv,comp8_rate_percent_diff,click_bool,gross_bookings_usd,booking_bool
0,1,2013-04-04 08:32:15,12,187,NaN,NaN,219,893,3,3.5,...,NaN,NaN,NaN,NaN,0.0,0.0,NaN,0,NaN,0
1,1,2013-04-04 08:32:15,12,187,NaN,NaN,219,10404,4,4.0,...,NaN,NaN,NaN,NaN,0.0,0.0,NaN,0,NaN,0
2,1,2013-04-04 08:32:15,12,187,NaN,NaN,219,21315,3,4.5,...,NaN,NaN,NaN,NaN,0.0,0.0,NaN,0,NaN,0
3,1,2013-04-04 08:32:15,12,187,NaN,NaN,219,27348,2,4.0,...,NaN,NaN,NaN,NaN,-1.0,0.0,5.0,0,NaN,0
4,1,2013-04-04 08:32:15,12,187,NaN,NaN,219,29604,4,3.5,...,NaN,NaN,NaN,NaN,0.0,0.0,NaN,0,NaN,0


compute target column relevance

In [3]:
# Relevance follows the official NDCG logic: booking = 5, click-only = 1, ignored = 0
train["relevance"] = np.where(
    train["booking_bool"] == 1, 5,
    np.where(train["click_bool"] == 1, 1, 0)
).astype(np.int8)

Split train into train/validation by time and complete search groups

In [4]:
train["date_time"] = pd.to_datetime(train["date_time"])

search_dates = train.groupby("srch_id")["date_time"].min()
cutoff = search_dates.quantile(0.80)

train_srch_ids = search_dates[search_dates < cutoff].index
val_srch_ids = search_dates[search_dates >= cutoff].index

train_df = train[train["srch_id"].isin(train_srch_ids)].copy()
val_df = train[train["srch_id"].isin(val_srch_ids)].copy()

print("Train fold shape:", train_df.shape)
print("Validation fold shape:", val_df.shape)
print("Train searches:", train_df["srch_id"].nunique())
print("Validation searches:", val_df["srch_id"].nunique())

Train fold shape: (3980039, 55)
Validation fold shape: (978308, 55)
Train searches: 159836
Validation searches: 39959


Memory reduction helper -> Check if we still want this later on!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!

In [5]:

def reduce_memory_usage(df):
    
    df = df.copy()

    for col in df.columns:
        if pd.api.types.is_integer_dtype(df[col]):
            df[col] = pd.to_numeric(df[col], downcast="integer")

        elif pd.api.types.is_float_dtype(df[col]):
            df[col] = pd.to_numeric(df[col], downcast="float")

    return df



Fit preprocessing values on training fold only

In [6]:
def compute_impute_vals(df):
    """
    Computes imputation values and caps using only the training fold.
    These values are then reused for validation and test, preventing leakage.
    """
    params = {}

    params["visitor_hist_starrating_fill"] = -1
    params["visitor_hist_adr_usd_fill"] = -1

    # srch_query_affinity_score is a negative log probability.
    # Missing gets a value slightly below the minimum observed value.
    params["srch_query_affinity_score_fill"] = df["srch_query_affinity_score"].min(skipna=True) - 1

    params["orig_destination_distance_fill"] = df["orig_destination_distance"].median()
    params["prop_location_score2_fill"] = df["prop_location_score2"].median()
    params["prop_review_score_fill"] = df["prop_review_score"].median()

    # Price cap based on training only.
    # We cap extreme outliers but still keep price information.
    params["price_usd_cap"] = df["price_usd"].quantile(0.999)

    return params


impute_values = compute_impute_vals(train_df)

Basic cleaning, missing indicators, imputations, and simple transformations

In [7]:
def add_basic_features(df, params):
    """
    Adds basic non-target features:
    - missing indicators
    - missing-value imputations
    - price transformations
    - review/star zero indicators
    - trip/user context features
    - date/time features
    """
    df = df.copy()

    # Missing indicators
    df["visitor_hist_starrating_missing"] = df["visitor_hist_starrating"].isna().astype(np.int8)
    df["visitor_hist_adr_usd_missing"] = df["visitor_hist_adr_usd"].isna().astype(np.int8)
    df["srch_query_affinity_score_missing"] = df["srch_query_affinity_score"].isna().astype(np.int8)
    df["orig_destination_distance_missing"] = df["orig_destination_distance"].isna().astype(np.int8)
    df["prop_location_score2_missing"] = df["prop_location_score2"].isna().astype(np.int8)
    df["prop_review_score_missing"] = df["prop_review_score"].isna().astype(np.int8)

    # Special zero indicators
    # prop_review_score = 0 means no reviews.
    # prop_starrating = 0 means unknown/no stars/not publicized.
    df["prop_review_score_is_zero"] = (df["prop_review_score"] == 0).astype(np.int8)
    df["prop_starrating_is_zero"] = (df["prop_starrating"] == 0).astype(np.int8)


    # Impute missing values
    df["visitor_hist_starrating"] = df["visitor_hist_starrating"].fillna(params["visitor_hist_starrating_fill"])
    df["visitor_hist_adr_usd"] = df["visitor_hist_adr_usd"].fillna(params["visitor_hist_adr_usd_fill"])
    df["srch_query_affinity_score"] = df["srch_query_affinity_score"].fillna(params["srch_query_affinity_score_fill"])
    df["orig_destination_distance"] = df["orig_destination_distance"].fillna(params["orig_destination_distance_fill"])
    df["prop_location_score2"] = df["prop_location_score2"].fillna(params["prop_location_score2_fill"])
    df["prop_review_score"] = df["prop_review_score"].fillna(params["prop_review_score_fill"])

   
    # Low/zero price indicators
    df["price_is_zero"] = (df["price_usd"] == 0).astype(np.int8)
    df["price_below_1"] = ((df["price_usd"] > 0) & (df["price_usd"] < 1)).astype(np.int8)
    df["price_below_10"] = ((df["price_usd"] > 0) & (df["price_usd"] < 10)).astype(np.int8)

  
    # We create multiple versions of price to let the model learn which works best:
    # - capped: upper extreme outliers are capped
    # - clipped: upper and lower extreme outliers are clipped
    df["price_usd_capped"] = df["price_usd"].clip(upper=params["price_usd_cap"])
    df["price_usd_clipped"] = df["price_usd"].clip(lower=1, upper=params["price_usd_cap"])

    df["log_price_usd"] = np.log1p(df["price_usd"])
    df["log_price_usd_capped"] = np.log1p(df["price_usd_capped"])
    df["log_price_usd_clipped"] = np.log1p(df["price_usd_clipped"])

    
    df["log_orig_destination_distance"] = np.log1p(df["orig_destination_distance"])

    # Trip/user context features
    # Guest info: whether children are present, whether multiple rooms are booked
    df["total_guests"] = df["srch_adults_count"] + df["srch_children_count"]
    df["children_present"] = (df["srch_children_count"] > 0).astype(np.int8)
    df["multi_room"] = (df["srch_room_count"] > 1).astype(np.int8)

    # Domestic search, whether the searcher is from the same country as the hotel
    df["domestic_search"] = (
        df["visitor_location_country_id"] == df["prop_country_id"]
    ).astype(np.int8)

    # Booking behaviour indicators
    df["log_booking_window"] = np.log1p(df["srch_booking_window"])
    df["log_length_of_stay"] = np.log1p(df["srch_length_of_stay"])
    
    df["last_minute_booking"] = (df["srch_booking_window"] <= 1).astype(np.int8)
    df["long_booking_window"] = (df["srch_booking_window"] >= 90).astype(np.int8)
    df["long_stay"] = (df["srch_length_of_stay"] >= 7).astype(np.int8)

    # Star/review, catch non-linear relationships
    df["is_4_star"] = (df["prop_starrating"] == 4).astype(np.int8)
    df["is_5_star"] = (df["prop_starrating"] == 5).astype(np.int8)
    df["high_review_score"] = (df["prop_review_score"] >= 4.0).astype(np.int8)

    # Date/time features
    df["date_time"] = pd.to_datetime(df["date_time"])
    df["search_month"] = df["date_time"].dt.month.astype(np.int8)
    df["search_quarter"] = (((df["search_month"] - 1) // 3) + 1).astype(np.int8)
    df["search_dayofweek"] = df["date_time"].dt.dayofweek.astype(np.int8)
    df["search_hour"] = df["date_time"].dt.hour.astype(np.int8)
    df["search_is_weekend"] = df["search_dayofweek"].isin([5, 6]).astype(np.int8)

    return df

Competitor missingness and aggregate competitor features

In [8]:
comp_rate_cols = [f"comp{i}_rate" for i in range(1, 9)]
comp_inv_cols = [f"comp{i}_inv" for i in range(1, 9)]
comp_pct_cols = [f"comp{i}_rate_percent_diff" for i in range(1, 9)]

def add_competitor_features(df):
    """
    Adds competitor features.
    The raw competitor columns are very sparse, so we summarize them with counts.
    """
    df = df.copy()

    # Missing-count features before filling
    df["comp_rate_missing_count"] = df[comp_rate_cols].isna().sum(axis=1).astype(np.int8)
    df["comp_inv_missing_count"] = df[comp_inv_cols].isna().sum(axis=1).astype(np.int8)
    df["comp_pct_missing_count"] = df[comp_pct_cols].isna().sum(axis=1).astype(np.int8)

  
    # Any competitor data available?
    df["comp_has_any_rate_data"] = (df["comp_rate_missing_count"] < 8).astype(np.int8)
    df["comp_has_any_inv_data"] = (df["comp_inv_missing_count"] < 8).astype(np.int8)
    df["comp_has_any_pct_data"] = (df["comp_pct_missing_count"] < 8).astype(np.int8)

  
    # Aggregate features before filling
    df["comp_expedia_cheaper_count"] = df[comp_rate_cols].eq(1).sum(axis=1).astype(np.int8)
    df["comp_expedia_more_expensive_count"] = df[comp_rate_cols].eq(-1).sum(axis=1).astype(np.int8)
    df["comp_same_price_count"] = df[comp_rate_cols].eq(0).sum(axis=1).astype(np.int8)

    df["comp_unavailable_count"] = df[comp_inv_cols].eq(1).sum(axis=1).astype(np.int8)
    df["comp_available_count"] = df[comp_inv_cols].eq(0).sum(axis=1).astype(np.int8)

    # Missing flags and fill raw competitor columns
    for i in range(1, 9):
        rate_col = f"comp{i}_rate"
        inv_col = f"comp{i}_inv"
        pct_col = f"comp{i}_rate_percent_diff"

        df[f"{rate_col}_missing"] = df[rate_col].isna().astype(np.int8)
        df[f"{inv_col}_missing"] = df[inv_col].isna().astype(np.int8)
        df[f"{pct_col}_missing"] = df[pct_col].isna().astype(np.int8)

        # -2 means no competitor data.
        df[rate_col] = df[rate_col].fillna(-2)
        df[inv_col] = df[inv_col].fillna(-2)

        # For percent difference, missing means no known difference.
        # We already captured missingness separately, so fill with 0.
        df[pct_col] = df[pct_col].fillna(0)

    df["comp_no_rate_data_count"] = df[comp_rate_cols].eq(-2).sum(axis=1).astype(np.int8)
    df["comp_no_inv_data_count"] = df[comp_inv_cols].eq(-2).sum(axis=1).astype(np.int8)

    return df

Within-search relative features

In [9]:

def add_within_search_features(df):
    """
    Adds features comparing each hotel to the other hotels in the same search.
    These could be valuable because the task is to rank hotels within srch_id.
    """
    df = df.copy()
    group = df.groupby("srch_id")

    df["hotels_in_search"] = group["prop_id"].transform("count")

    
    # Price relative to search
    df["search_price_mean"] = group["price_usd_capped"].transform("mean")
    df["search_price_median"] = group["price_usd_capped"].transform("median")
    df["search_price_min"] = group["price_usd_capped"].transform("min")
    df["search_price_max"] = group["price_usd_capped"].transform("max")

    df["price_diff_from_search_mean"] = df["price_usd_capped"] - df["search_price_mean"]
    df["price_diff_from_search_median"] = df["price_usd_capped"] - df["search_price_median"]
    df["price_diff_from_search_min"] = df["price_usd_capped"] - df["search_price_min"]

    df["price_ratio_to_search_mean"] = df["price_usd_capped"] / (df["search_price_mean"] + 1e-6)
    df["price_ratio_to_search_median"] = df["price_usd_capped"] / (df["search_price_median"] + 1e-6)

    # Lower price gets rank 1.
    df["price_rank_in_search"] = group["price_usd_capped"].rank(method="average", ascending=True)
    df["price_pct_rank_in_search"] = group["price_usd_capped"].rank(method="average", pct=True, ascending=True)

    
    # Log price relative to search
    df["search_log_price_mean"] = group["log_price_usd_capped"].transform("mean")
    df["log_price_diff_from_search_mean"] = df["log_price_usd_capped"] - df["search_log_price_mean"]

   
    # Star rating relative to search
    df["search_star_mean"] = group["prop_starrating"].transform("mean")
    df["search_star_max"] = group["prop_starrating"].transform("max")

    df["star_diff_from_search_mean"] = df["prop_starrating"] - df["search_star_mean"]
    df["star_diff_from_search_max"] = df["prop_starrating"] - df["search_star_max"]

    # Higher star rating gets rank 1.
    df["star_rank_in_search"] = group["prop_starrating"].rank(method="average", ascending=False)
    df["star_pct_rank_in_search"] = group["prop_starrating"].rank(method="average", pct=True, ascending=False)

   
    # Review score relative to search
    df["search_review_mean"] = group["prop_review_score"].transform("mean")
    df["search_review_max"] = group["prop_review_score"].transform("max")

    df["review_diff_from_search_mean"] = df["prop_review_score"] - df["search_review_mean"]
    df["review_diff_from_search_max"] = df["prop_review_score"] - df["search_review_max"]

    # Higher review gets rank 1.
    df["review_rank_in_search"] = group["prop_review_score"].rank(method="average", ascending=False)
    df["review_pct_rank_in_search"] = group["prop_review_score"].rank(method="average", pct=True, ascending=False)

    # Location score relative to search
    df["search_location1_mean"] = group["prop_location_score1"].transform("mean")
    df["search_location2_mean"] = group["prop_location_score2"].transform("mean")

    df["location1_diff_from_search_mean"] = df["prop_location_score1"] - df["search_location1_mean"]
    df["location2_diff_from_search_mean"] = df["prop_location_score2"] - df["search_location2_mean"]

    # Higher location score gets rank 1.
    df["location2_rank_in_search"] = group["prop_location_score2"].rank(method="average", ascending=False)
    df["location2_pct_rank_in_search"] = group["prop_location_score2"].rank(method="average", pct=True, ascending=False)

    # Promotion context
    df["search_promotion_rate"] = group["promotion_flag"].transform("mean")
    df["promotion_above_search_avg"] = df["promotion_flag"] - df["search_promotion_rate"]

    return df

Property and destination aggregate features

In [10]:
def compute_aggregate_features(df):
    """
    Computes safe aggregation features using non-target columns only.
    """

    df = df.copy()

    # fjern dette etter å ha kjørt gjennom en gang for å se at flyten går
    if "price_usd_capped" not in df.columns:
        raise ValueError("Run add_basic_features before fit_aggregate_features.")

    # Prop-id aggregates
    prop_agg = df.groupby("prop_id").agg(
        prop_id_count=("prop_id", "size"),
        prop_id_mean_price=("price_usd_capped", "mean"),
        prop_id_median_price=("price_usd_capped", "median"),
        prop_id_std_price=("price_usd_capped", "std"),
        prop_id_mean_log_price=("log_price_usd_capped", "mean"),
        prop_id_mean_starrating=("prop_starrating", "mean"),
        prop_id_mean_review_score=("prop_review_score", "mean"),
        prop_id_mean_location_score1=("prop_location_score1", "mean"),
        prop_id_mean_location_score2=("prop_location_score2", "mean"),
        prop_id_mean_promotion_flag=("promotion_flag", "mean")
    ).reset_index()

    prop_agg["prop_id_std_price"] = prop_agg["prop_id_std_price"].fillna(0)


    # Destination aggregates
    dest_agg = df.groupby("srch_destination_id").agg(
        dest_id_count=("srch_destination_id", "size"),
        dest_id_mean_price=("price_usd_capped", "mean"),
        dest_id_median_price=("price_usd_capped", "median"),
        dest_id_mean_starrating=("prop_starrating", "mean"),
        dest_id_mean_review_score=("prop_review_score", "mean"),
        dest_id_mean_location_score2=("prop_location_score2", "mean")
    ).reset_index()

   
    # Property-destination appearance count
    # This captures how often a property appears for a destination.
    prop_dest_agg = df.groupby(["prop_id", "srch_destination_id"]).agg(
        prop_dest_count=("prop_id", "size")
    ).reset_index()

    return prop_agg, dest_agg, prop_dest_agg


def add_aggregate_features(df, prop_agg, dest_agg, prop_dest_agg):
    """
    Merges the aggregated features into train, only on already existing prop_ids and destinations.
    """
    df = df.copy()

    df = df.merge(prop_agg, on="prop_id", how="left")
    df = df.merge(dest_agg, on="srch_destination_id", how="left")
    df = df.merge(prop_dest_agg, on=["prop_id", "srch_destination_id"], how="left")
    return df


def fill_aggregate_missing_values(df_train, df_test):
    """
    For validation and test set there will be missing aggregate values for unseen properties/destinations.
    Fills aggregate features that are missing using medians from train dataframe after merging.
    """
    df_test = df_test.copy()
    agg_cols = [
        col for col in df_test.columns
        if col.startswith("prop_id_") or col.startswith("dest_id_") or col.startswith("prop_dest_")
    ]

    for col in agg_cols:
        if col in df_train.columns and df_test[col].isna().any():
            fill_value = df_train[col].median()
            df_test[col] = df_test[col].fillna(fill_value)

    return df_test

Add all features and prepare datasets

In [11]:
def add_features_pipeline(df, params):
    df = add_basic_features(df, params)
    df = add_competitor_features(df)
    df = add_within_search_features(df)
    return df

In [12]:
train_no_agg = add_features_pipeline(train_df, impute_values)
val_no_agg = add_features_pipeline(val_df, impute_values)

propid_agg, destid_agg, prop_dest_agg = compute_aggregate_features(train_no_agg)

train_features = add_aggregate_features(train_no_agg, propid_agg, destid_agg, prop_dest_agg)
val_features = add_aggregate_features(val_no_agg, propid_agg, destid_agg, prop_dest_agg)

val_features = fill_aggregate_missing_values(train_features, val_features)

In [13]:
print("Prepared train shape:", train_features.shape)
print("Prepared validation shape:", val_features.shape)

display(train_features.head())

Prepared train shape: (3980039, 177)
Prepared validation shape: (978308, 177)


,srch_id,date_time,site_id,visitor_location_country_id,visitor_hist_starrating,visitor_hist_adr_usd,prop_country_id,prop_id,prop_starrating,prop_review_score,...,prop_id_mean_location_score1,prop_id_mean_location_score2,prop_id_mean_promotion_flag,dest_id_count,dest_id_mean_price,dest_id_median_price,dest_id_mean_starrating,dest_id_mean_review_score,dest_id_mean_location_score2,prop_dest_count
0,1,2013-04-04 08:32:15,12,187,-1.0,-1.0,219,893,3,3.5,...,2.83,0.057397,0.087824,10956,132.218604,117.0,3.158817,3.788335,0.040318,254
1,1,2013-04-04 08:32:15,12,187,-1.0,-1.0,219,10404,4,4.0,...,2.20,0.028304,0.061856,10956,132.218604,117.0,3.158817,3.788335,0.040318,230
2,1,2013-04-04 08:32:15,12,187,-1.0,-1.0,219,21315,3,4.5,...,2.20,0.042165,0.000000,10956,132.218604,117.0,3.158817,3.788335,0.040318,240
3,1,2013-04-04 08:32:15,12,187,-1.0,-1.0,219,27348,2,4.0,...,2.83,0.024148,0.180693,10956,132.218604,117.0,3.158817,3.788335,0.040318,191
4,1,2013-04-04 08:32:15,12,187,-1.0,-1.0,219,29604,4,3.5,...,2.64,0.138880,0.204668,10956,132.218604,117.0,3.158817,3.788335,0.040318,246


In [14]:
train_features = reduce_memory_usage(train_features)
val_features = reduce_memory_usage(val_features)

In [15]:
train_features.to_parquet("data/train_features.parquet", index=False)
val_features.to_parquet("data/val_features.parquet", index=False)

In [ ]:
full_train_impute_vals = compute_impute_vals(train)
full_train_features = add_features_pipeline(train, full_train_impute_vals)
full_propid_agg, full_destid_agg, full_prop_dest_agg = compute_aggregate_features(full_train_features)
full_train_features = add_aggregate_features(full_train_features, full_propid_agg, full_destid_agg, full_prop_dest_agg)

full_test_features = add_features_pipeline(test_df, full_train_impute_vals)
full_test_features = add_aggregate_features(full_test_features, full_propid_agg, full_destid_agg, full_prop_dest_agg)
full_test_features = fill_aggregate_missing_values(full_train_features, full_test_features)
print("Full train prepared shape:", full_train_features.shape)
print("Full test prepared shape:", full_test_features.shape)

print("Full train prepared shape:", full_train_features.shape)
print("Full test prepared shape:", full_test_features.shape)

full_train_features.to_parquet("data/full_train_features.parquet", index=False)
full_test_features.to_parquet("data/full_test_features.parquet", index=False) 

Full train prepared shape: (4958347, 177)
Full test prepared shape: (4959183, 172)
Full train prepared shape: (4958347, 177)
Full test prepared shape: (4959183, 172)
